<a href="https://colab.research.google.com/github/ChiBinh123/NhapMonHocMay/blob/main/Tu%E1%BA%A7n5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from math import log2

# =========================================================
# 1. DỮ LIỆU PLAY TENNIS
# =========================================================

data = {
    "outlook": [
        "sunny", "sunny", "overcast", "rainy",
        "rainy", "rainy", "overcast", "sunny",
        "sunny", "rainy", "sunny", "overcast",
        "overcast", "rainy"
    ],

    "temp": [
        "hot", "hot", "hot", "mild",
        "cool", "cool", "cool", "mild",
        "cool", "mild", "mild", "mild",
        "hot", "mild"
    ],

    "humidity": [
        "high", "high", "high", "high",
        "normal", "normal", "normal", "high",
        "normal", "normal", "normal", "high",
        "normal", "high"
    ],

    "windy": [
        "false", "true", "false", "false",
        "false", "true", "true", "false",
        "false", "false", "true", "true",
        "false", "true"
    ],

    "play": [
        "no", "no", "yes", "yes",
        "yes", "no", "yes", "no",
        "yes", "yes", "yes", "yes",
        "yes", "no"
    ]
}

df = pd.DataFrame(data)

print("=" * 60)
print("DỮ LIỆU PLAY TENNIS")
print("=" * 60)
display(df)


# =========================================================
# 2. HÀM TÍNH ENTROPY
# =========================================================

def entropy(y):

    values, counts = np.unique(y, return_counts=True)

    total = len(y)

    ent = 0

    for count in counts:

        p = count / total

        if p > 0:
            ent -= p * log2(p)

    return ent


# =========================================================
# 3. HÀM TÍNH INFORMATION GAIN
# =========================================================

def information_gain(data, attribute, target):

    # Entropy trước khi chia
    entropy_before = entropy(data[target])

    # Entropy sau khi chia
    weighted_entropy = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / len(data)

        weighted_entropy += weight * entropy(subset[target])

    # Gain
    gain = entropy_before - weighted_entropy

    return gain


# =========================================================
# 4. TÍNH GAIN CHO CÁC THUỘC TÍNH
# =========================================================

def show_gains(data, attributes, target):

    print("\nCác Information Gain:")

    gains = {}

    for attribute in attributes:

        gain = information_gain(
            data,
            attribute,
            target
        )

        gains[attribute] = gain

        print(
            f"Gain({attribute}) = {gain:.3f}"
        )

    return gains


# =========================================================
# 5. THUẬT TOÁN ID3
# =========================================================

def id3(data, attributes, target, level=0):

    # Khoảng cách để in cây đẹp hơn
    space = "    " * level

    # Các giá trị của lớp mục tiêu
    target_values = data[target].unique()

    # -----------------------------------------------------
    # Trường hợp 1: Tất cả đều YES
    # -----------------------------------------------------

    if len(target_values) == 1:

        result = target_values[0]

        print(
            f"{space}=> {result}"
        )

        return result

    # -----------------------------------------------------
    # Trường hợp 2: Không còn thuộc tính
    # -----------------------------------------------------

    if len(attributes) == 0:

        result = data[target].mode()[0]

        print(
            f"{space}=> {result}"
        )

        return result

    # -----------------------------------------------------
    # Tính Gain
    # -----------------------------------------------------

    gains = show_gains(
        data,
        attributes,
        target
    )

    # Chọn Gain lớn nhất
    best_attribute = max(
        gains,
        key=gains.get
    )

    print()
    print(
        f"{space}Chọn: {best_attribute}"
        f"  (Gain = {gains[best_attribute]:.3f})"
    )

    # Tạo cây
    tree = {
        best_attribute: {}
    }

    # Các thuộc tính còn lại
    remaining_attributes = [
        a for a in attributes
        if a != best_attribute
    ]

    # -----------------------------------------------------
    # Chia dữ liệu theo thuộc tính được chọn
    # -----------------------------------------------------

    for value in data[best_attribute].unique():

        print()
        print(
            f"{space}[ {best_attribute} = {value} ]"
        )

        subset = data[
            data[best_attribute] == value
        ]

        # Gọi đệ quy
        subtree = id3(
            subset,
            remaining_attributes,
            target,
            level + 1
        )

        tree[best_attribute][value] = subtree

    return tree


# =========================================================
# 6. CHẠY ID3
# =========================================================

attributes = [
    "outlook",
    "temp",
    "humidity",
    "windy"
]

print("\n")
print("=" * 60)
print("BẮT ĐẦU THUẬT TOÁN ID3")
print("=" * 60)

print(
    "\nEntropy(S) =",
    round(entropy(df["play"]), 3)
)

tree = id3(
    df,
    attributes,
    "play"
)


# =========================================================
# 7. IN CÂY CUỐI CÙNG
# =========================================================

print("\n")
print("=" * 60)
print("CÂY QUYẾT ĐỊNH ID3")
print("=" * 60)


def print_tree(tree, indent=""):

    if not isinstance(tree, dict):
        print(indent + "=> " + str(tree))
        return

    attribute = list(tree.keys())[0]

    print(indent + attribute)

    branches = tree[attribute]

    for value, subtree in branches.items():

        print(
            indent
            + "├── "
            + str(value)
        )

        print_tree(
            subtree,
            indent + "│   "
        )


print_tree(tree)

DỮ LIỆU PLAY TENNIS


,outlook,temp,humidity,windy,play
0,sunny,hot,high,false,no
1,sunny,hot,high,true,no
2,overcast,hot,high,false,yes
3,rainy,mild,high,false,yes
4,rainy,cool,normal,false,yes
5,rainy,cool,normal,true,no
6,overcast,cool,normal,true,yes
7,sunny,mild,high,false,no
8,sunny,cool,normal,false,yes
9,rainy,mild,normal,false,yes




BẮT ĐẦU THUẬT TOÁN ID3

Entropy(S) = 0.94

Các Information Gain:
Gain(outlook) = 0.247
Gain(temp) = 0.029
Gain(humidity) = 0.152
Gain(windy) = 0.048

Chọn: outlook  (Gain = 0.247)

[ outlook = sunny ]

Các Information Gain:
Gain(temp) = 0.571
Gain(humidity) = 0.971
Gain(windy) = 0.020

    Chọn: humidity  (Gain = 0.971)

    [ humidity = high ]
        => no

    [ humidity = normal ]
        => yes

[ outlook = overcast ]
    => yes

[ outlook = rainy ]

Các Information Gain:
Gain(temp) = 0.020
Gain(humidity) = 0.020
Gain(windy) = 0.971

    Chọn: windy  (Gain = 0.971)

    [ windy = false ]
        => yes

    [ windy = true ]
        => no


CÂY QUYẾT ĐỊNH ID3
outlook
├── sunny
│   humidity
│   ├── high
│   │   => no
│   ├── normal
│   │   => yes
├── overcast
│   => yes
├── rainy
│   windy
│   ├── false
│   │   => yes
│   ├── true
│   │   => no
